# Tahap 2 - Case Representation

Project: Case-Based Reasoning untuk Pidana Umum - Pencurian di PN Tangerang

Notebook ini digunakan sebagai bagian dari pipeline CBR.


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")

txt_files = list(RAW_DIR.glob("*.txt"))

print("Jumlah data di inventory:", len(df_inventory))
print("Jumlah file .txt di data/raw:", len(txt_files))

df_inventory[["case_id", "no_perkara", "tanggal_putusan", "status_download", "jumlah_kata", "raw_file"]].head(40)

Jumlah data di inventory: 40
Jumlah file .txt di data/raw: 40


,case_id,no_perkara,tanggal_putusan,status_download,jumlah_kata,raw_file
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,berhasil_tapi_teks_pendek,33,case_001.txt
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,berhasil_tapi_teks_pendek,41,case_002.txt
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,berhasil_tapi_teks_pendek,33,case_003.txt
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,berhasil_tapi_teks_pendek,41,case_004.txt
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,berhasil_tapi_teks_pendek,41,case_005.txt
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,berhasil_tapi_teks_pendek,41,case_006.txt
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,berhasil_tapi_teks_pendek,41,case_007.txt
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,berhasil_tapi_teks_pendek,41,case_008.txt
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,berhasil_tapi_teks_pendek,41,case_009.txt
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,berhasil_tapi_teks_pendek,41,case_010.txt


In [2]:
import re
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

inventory_path = PROCESSED_DIR / "case_inventory.csv"
cases_path = PROCESSED_DIR / "cases.csv"

df_inventory = pd.read_csv(inventory_path, dtype=str).fillna("")


def clean_space(text):
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    return text.strip()


def extract_no_perkara(text):
    patterns = [
        r"Nomor\s*[:\-]?\s*([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"Nomor\s*[:\-]?\s*([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
        r"([0-9]+\/Pid\.B\/[0-9]{4}\/PN\.?\s*Tng)",
        r"([0-9]+\/PID\.B\/[0-9]{4}\/PN\.?\s*TNG)",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1).replace(" ", "")
    
    return ""


def extract_tanggal_putusan(text):
    bulan = "Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember"

    patterns = [
        rf"tanggal\s+([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"pada\s+hari\s+.*?tanggal\s+([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
        rf"([0-9]{{1,2}}\s+(?:{bulan})\s+[0-9]{{4}})",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    
    return ""


def extract_pasal(text):
    patterns = [
        r"Pasal\s+[0-9]+[A-Za-z]?(?:\s+ayat\s+\([0-9]+\))?(?:\s+ke[-\s]?[0-9]+)?(?:\s+KUHP)?",
        r"Pasal\s+[0-9]+[A-Za-z]?\s+KUHP",
        r"Pasal\s+[0-9]+[A-Za-z]?"
    ]

    hasil = []

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for m in matches:
            m = " ".join(m.split())
            if m not in hasil:
                hasil.append(m)

    return "; ".join(hasil[:10])


def extract_terdakwa(text):
    patterns = [
        r"Terdakwa\s*[:\-]?\s*(.+?)(?:\n|Penuntut|Tanggal|Mengadili)",
        r"Nama\s+lengkap\s*[:\-]?\s*(.+?)(?:\n|Tempat|Umur|Jenis)",
        r"atas\s+nama\s*[:\-]?\s*(.+?)(?:\n|,)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            hasil = match.group(1).strip()
            hasil = re.sub(r"\s+", " ", hasil)
            return hasil[:200]
    
    return ""


def extract_amar_putusan(text):
    lower_text = text.lower()

    start = lower_text.find("mengadili")
    
    if start == -1:
        start = lower_text.find("m e n g a d i l i")
    
    if start == -1:
        return ""

    amar = text[start:start + 2500]
    amar = clean_space(amar)

    return amar


def extract_ringkasan_fakta(text):
    lower_text = text.lower()

    start = lower_text.find("menimbang")
    if start == -1:
        start = lower_text.find("bahwa")
    if start == -1:
        start = 0

    ringkasan = text[start:start + 2000]
    ringkasan = clean_space(ringkasan)

    return ringkasan


def extract_argumen_hukum(text):
    lower_text = text.lower()

    keywords = ["menimbang", "pasal", "terbukti", "unsur", "kuhp"]
    positions = []

    for key in keywords:
        pos = lower_text.find(key)
        if pos != -1:
            positions.append(pos)

    if not positions:
        return ""

    start = min(positions)
    argumen = text[start:start + 2000]
    argumen = clean_space(argumen)

    return argumen


cases = []

for _, row in df_inventory.iterrows():
    case_id = row.get("case_id", "")
    raw_file = row.get("raw_file", "")
    raw_path = RAW_DIR / raw_file

    if not raw_path.exists():
        print(f"File tidak ditemukan: {raw_file}")
        continue

    text = raw_path.read_text(encoding="utf-8", errors="ignore")
    text = clean_space(text)

    jumlah_kata = len(text.split())

    no_perkara = row.get("no_perkara", "")
    if no_perkara == "":
        no_perkara = extract_no_perkara(text)

    tanggal_putusan = row.get("tanggal_putusan", "")
    if tanggal_putusan == "":
        tanggal_putusan = extract_tanggal_putusan(text)

    pasal = extract_pasal(text)
    terdakwa = extract_terdakwa(text)
    amar_putusan = extract_amar_putusan(text)
    ringkasan_fakta = extract_ringkasan_fakta(text)
    argumen_hukum = extract_argumen_hukum(text)

    cases.append({
        "case_id": case_id,
        "no_perkara": no_perkara,
        "tanggal_putusan": tanggal_putusan,
        "pengadilan": row.get("pengadilan", "PN Tangerang"),
        "jenis_perkara": row.get("jenis_perkara", "Pidana Umum - Pencurian"),
        "pasal": pasal,
        "terdakwa": terdakwa,
        "amar_putusan": amar_putusan,
        "ringkasan_fakta": ringkasan_fakta,
        "argumen_hukum": argumen_hukum,
        "sumber_url": row.get("sumber_url", ""),
        "raw_file": raw_file,
        "jumlah_kata": jumlah_kata,
        "text_full": text
    })

cases_df = pd.DataFrame(cases)

cases_df.to_csv(cases_path, index=False)

print("cases.csv berhasil dibuat.")
print("Lokasi:", cases_path)
print("Jumlah kasus:", len(cases_df))

cases_df.head()

cases.csv berhasil dibuat.
Lokasi: /Users/bagasfernanda/Desktop/cbr-pencurian-pn-tangerang/data/processed/cases.csv
Jumlah kasus: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,pasal,terdakwa,amar_putusan,ringkasan_fakta,argumen_hukum,sumber_url,raw_file,jumlah_kata,text_full
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,PN Tangerang,Pidana Umum - Pencurian,,,,bahwa Anda bukan bot.\nRay ID: a0ecb0eb2e05ea7...,,https://putusan3.mahkamahagung.go.id/direktori...,case_001.txt,33,putusan3.mahkamahagung.go.id\nMelakukan verifi...
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,PN Tangerang,Pidana Umum - Pencurian,,,,bahwa Anda adalah manusia. Ini dapat memerluka...,,https://putusan3.mahkamahagung.go.id/direktori...,case_002.txt,41,putusan3.mahkamahagung.go.id\nVerifikasi bahwa...
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,PN Tangerang,Pidana Umum - Pencurian,,,,bahwa Anda bukan bot.\nRay ID: a0ecb1489dade77...,,https://putusan3.mahkamahagung.go.id/direktori...,case_003.txt,33,putusan3.mahkamahagung.go.id\nMelakukan verifi...
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,PN Tangerang,Pidana Umum - Pencurian,,,,bahwa Anda adalah manusia. Ini dapat memerluka...,,https://putusan3.mahkamahagung.go.id/direktori...,case_004.txt,41,putusan3.mahkamahagung.go.id\nVerifikasi bahwa...
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,PN Tangerang,Pidana Umum - Pencurian,,,,bahwa Anda adalah manusia. Ini dapat memerluka...,,https://putusan3.mahkamahagung.go.id/direktori...,case_005.txt,41,putusan3.mahkamahagung.go.id\nVerifikasi bahwa...


In [3]:
cases_df = pd.read_csv(cases_path, dtype=str).fillna("")

print("Jumlah kasus:", len(cases_df))

cases_df[[
    "case_id",
    "no_perkara",
    "tanggal_putusan",
    "pengadilan",
    "jenis_perkara",
    "pasal",
    "terdakwa",
    "jumlah_kata"
]].head(40)

Jumlah kasus: 40


,case_id,no_perkara,tanggal_putusan,pengadilan,jenis_perkara,pasal,terdakwa,jumlah_kata
0,case_001,2184/PID.B/2013/PN.TNG,07-01-2014,PN Tangerang,Pidana Umum - Pencurian,,,33
1,case_002,1671/Pid.B/2021/PN.Tng,18-10-2021,PN Tangerang,Pidana Umum - Pencurian,,,41
2,case_003,1885/Pid.B/2022/PN.Tng,10-10-2022,PN Tangerang,Pidana Umum - Pencurian,,,33
3,case_004,1314/PID.B/2013/PN.TNG,24-07-2013,PN Tangerang,Pidana Umum - Pencurian,,,41
4,case_005,1493/PID.B/2013/PN.TNG,27-08-2013,PN Tangerang,Pidana Umum - Pencurian,,,41
5,case_006,310/Pid.B/2019/PN.Tng,18-03-2019,PN Tangerang,Pidana Umum - Pencurian,,,41
6,case_007,2686/Pid.B/2018/PN Tng,30 Januari 2019,PN Tangerang,Pidana Umum - Pencurian,,,41
7,case_008,1581/Pid.B/2011/PN.TNG,4 Oktober 2011,PN Tangerang,Pidana Umum - Pencurian,,,41
8,case_009,1022/Pid.B/2010/PN.TNG,14 Juli 2010,PN Tangerang,Pidana Umum - Pencurian,,,41
9,case_010,497 / PID.B / 2014 / PN.TNG.,19 Mei 2014,PN Tangerang,Pidana Umum - Pencurian,,,41
